# Paired Unstain ↔ H&E CycleGAN training at 2.0 MPP

This notebook retains two generators (`A→B`, `B→A`), two PatchGAN discriminators, cycle consistency, and identity learning. Filename-matched 2.0 MPP pairs receive identical spatial augmentation, and a mildly blurred paired L1 term is added in both directions to tolerate small residual registration error.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from cyclegan_core import (
    CycleGANTrainer, build_dataloaders, denormalize, seed_everything
)

## Parameters

The complete 2048×2048 source patch at 0.5 MPP covers 1024 µm. It is resized with bicubic antialiasing to 512×512, producing an exact 2.0 MPP training image with the same field of view.

In [ ]:
params = {
    'seed': 42,
    'gpu_index': 1,
    'data_dir': Path('../../data/HnE_n_UNStaining/patch_dataset_mpp05_2048'),
    'output_dir': Path('../../results/Unstain2HnE_paired_cyclegan_mpp20_v1'),
    'checkpoint_dir': Path('../../model/Unstain2HnE_paired_cyclegan_mpp20_v1'),
    'image_ext': 'png',
    'image_max_count': 30000,
    'original_size': 2048,
    'source_mpp': 0.5,
    'target_mpp': 2.0,
    'input_size': 512,       # full 2048px source view resized to 512px
    'paired_training': True,
    'batch_size': 2,         # two generators need more memory than the prior U-Net
    'num_epochs': 200,
    'decay_start_epoch': 100,
    'val_fraction': 0.10,
    'preload_images': True,
    'max_cache_gib': 64,
    'od_background_threshold': 0.98,
    'od_quantile': 0.995,
    'od_calibration_images': 256,
    # These crop settings are dormant when target_mpp=2.0 uses the full source patch.
    'min_crop_tissue_fraction': 0.10,
    'crop_retry_count': 20,
    'background_crop_probability': 0.10,
    'boundary_crop_probability': 0.20,
    'background_max_tissue_fraction': 0.02,
    'boundary_min_tissue_fraction': 0.02,
    'boundary_max_tissue_fraction': 0.50,
    'ngf': 32,
    'ndf': 64,
    'residual_blocks': 6,
    'lr': 2e-4,
    'beta1': 0.5,
    'beta2': 0.999,
    'lambda_cycle': 10.0,
    'lambda_identity': 5.0,
    'lambda_background': 10.0,
    'lambda_paired_blur': 5.0,
    'paired_blur_kernel': 5,
    'paired_blur_sigma': 0.8,
    'a_background_od_threshold': 0.04,
    'b_background_brightness_threshold': 0.94,
    'b_background_saturation_threshold': 0.06,
    'background_mask_blur_kernel': 5,
    'pool_size': 50,
    'preview_count': 2,
    'save_every': 10,
    # Start from newly initialized weights.
    'resume_checkpoint': None,
}

seed_everything(params['seed'])
if torch.cuda.is_available():
    device = torch.device(f"cuda:{params['gpu_index']}")
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device('cpu')
print('device:', device)

## Data

Both domains use a fixed `[-1, 1]` range. Domain A keeps the original grayscale optical-density direction and is repeated over three channels; domain B is normalized RGB H&E. Training uses the same filename and full field of view for A and B. Each 2048×2048 pair is resized to 512×512, and rotation/flip augmentation is applied identically to both images.

In [ ]:
train_loader, val_loader, od_max = build_dataloaders(params)

real_a, real_b = next(iter(train_loader))
fig, axes = plt.subplots(2, min(4, len(real_a)), figsize=(14, 7), squeeze=False)
for i in range(axes.shape[1]):
    axes[0, i].imshow(denormalize(real_a[i]).permute(1, 2, 0).numpy(), cmap='gray')
    axes[0, i].set_title('Domain A: OD [-1, 1]')
    axes[1, i].imshow(denormalize(real_b[i]).permute(1, 2, 0).numpy())
    axes[1, i].set_title('Paired Domain B: H&E')
    axes[0, i].axis('off')
    axes[1, i].axis('off')
plt.tight_layout()

## Models and training

The generator objective is `GAN(A→B) + GAN(B→A) + 10×cycle + 5×identity + 10×background + 5×weak paired blur`. Cycle learning remains fully active. The paired term uses a 5×5 Gaussian blur with sigma 0.8 before L1, reducing sensitivity to small residual registration error at 2.0 MPP.

In [ ]:
trainer = CycleGANTrainer(params, train_loader, val_loader, od_max, device)

In [ ]:
trainer.fit()

## Quick validation preview

`raw_paired_ssim` is printed only to observe translation quality. It is intentionally excluded from training and best-cycle checkpoint selection.

In [ ]:
metrics, preview = trainer.validate()
print(metrics)
trainer.save_preview(max(trainer.start_epoch - 1, 0), preview)